# 03 — Tuning by expanding-window cross-validation (months before 2021 only)

**Reads:** `data/processed/features.csv` (rows before January 2021 only).

**Writes:** `results/tables/cv_grid.csv`, `cv_best_by_feature_set.csv`, `cv_benchmarks.csv`,
`cv_choice.json`.

**Why not a random split.** Core CPI is persistent, so a month's neighbours carry its answer. A random
split puts months that come *after* a validation month into training. Every split here is in time order.

**The folds.** Four 24-month validation blocks: 2013–14, 2015–16, 2017–18, 2019–20. For each block the
model is trained on every month before it (from April 2005), so the training window grows block by block.

**What is chosen here, for each model:** the hyperparameters and the input set. The four input sets are core
history alone, plus used cars, plus jet fuel, plus both. An input set is kept only if it lowers the
cross-validated error compared with the same model without it: training error cannot judge an input,
because any extra column, even noise, lowers it.

**The score:** the average of the four blocks' RMSEs. Averaging per-block RMSEs keeps 2020 (the pandemic
months) to one block of four. The last cell checks whether the winner changes when the 2019–20 block is
left out.

The test months (January 2021 onward) are dropped in the first cell and never loaded here.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent
RAW = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
TAB = ROOT / "results" / "tables"
FIG = ROOT / "results" / "figures"
for d in (PROC, TAB, FIG):
    d.mkdir(parents=True, exist_ok=True)

import time
from nowcast_setup import (SEED, TEST_START, FEATURE_SETS, FOLDS, GRIDS, MODEL_LABELS,
                           make_model, load_features)

feat = load_features(PROC / "features.csv")
dev = feat.loc[: TEST_START - 1]                 # the test months stop here
print("development sample:", dev.index.min(), "to", dev.index.max(), f"({len(dev)} months)")
fold_names = [f"{a[:4]}-{b[2:4]}" for a, b in FOLDS]
for (a, b), name in zip(FOLDS, fold_names):
    print(f"block {name}: train {dev.index.min()}..{pd.Period(a, 'M') - 1} ({len(dev.loc[: pd.Period(a, 'M') - 1])} months), "
          f"validate {a}..{b} ({len(dev.loc[a:b])} months)")

development sample: 2005-04 to 2020-12 (189 months)
block 2013-14: train 2005-04..2012-12 (93 months), validate 2013-01..2014-12 (24 months)
block 2015-16: train 2005-04..2014-12 (117 months), validate 2015-01..2016-12 (24 months)
block 2017-18: train 2005-04..2016-12 (141 months), validate 2017-01..2018-12 (24 months)
block 2019-20: train 2005-04..2018-12 (165 months), validate 2019-01..2020-12 (24 months)


## Every model × input set × hyperparameter setting, scored on the four blocks

In [2]:
def rmse(e):
    return float(np.sqrt(np.mean(np.square(e))))


records = []
start_time = time.time()
for kind, grid in GRIDS.items():
    for fs_name, cols in FEATURE_SETS.items():
        for params in grid:
            scores = []
            for a, b in FOLDS:
                train = dev.loc[: pd.Period(a, "M") - 1]
                valid = dev.loc[a:b]
                model = make_model(kind, params).fit(train[cols], train["y"])
                scores.append(rmse(valid["y"] - model.predict(valid[cols])))
            records.append({"model": kind, "feature_set": fs_name, "params": json.dumps(params),
                            **dict(zip(fold_names, scores))})
    print(f"{MODEL_LABELS[kind]}: done after {time.time() - start_time:.0f} s")

cv = pd.DataFrame(records)
cv["cv_rmse"] = cv[fold_names].mean(axis=1)
cv["cv_rmse_without_2019_20"] = cv[fold_names[:3]].mean(axis=1)
cv.round(5).to_csv(TAB / "cv_grid.csv", index=False)
len(cv)

Random forest: done after 140 s


Gradient boosting (XGBoost): done after 159 s
Linear regression: done after 159 s


244

## Best setting for each model and input set

In [3]:
best = (cv.loc[cv.groupby(["model", "feature_set"])["cv_rmse"].idxmin()]
          .sort_values(["model", "cv_rmse"]).reset_index(drop=True))
best.round(4).to_csv(TAB / "cv_best_by_feature_set.csv", index=False)
best[["model", "feature_set", "params", "cv_rmse"] + fold_names].round(3)

,model,feature_set,params,cv_rmse,2013-14,2015-16,2017-18,2019-20
0,ols,core + used cars,{},0.101,0.076,0.084,0.087,0.158
1,ols,core + jet fuel,{},0.102,0.073,0.087,0.086,0.160
2,ols,core + used cars + jet fuel,{},0.102,0.073,0.086,0.089,0.158
3,ols,core,{},0.102,0.076,0.084,0.084,0.163
4,rf,core + jet fuel,"{""max_depth"": 2, ""min_samples_leaf"": 5, ""max_f...",0.105,0.077,0.081,0.085,0.177
5,rf,core,"{""max_depth"": 2, ""min_samples_leaf"": 5, ""max_f...",0.106,0.078,0.077,0.085,0.183
6,rf,core + used cars + jet fuel,"{""max_depth"": 2, ""min_samples_leaf"": 5, ""max_f...",0.106,0.079,0.082,0.084,0.178
7,rf,core + used cars,"{""max_depth"": 2, ""min_samples_leaf"": 20, ""max_...",0.106,0.076,0.079,0.086,0.183
8,xgb,core + jet fuel,"{""n_estimators"": 300, ""learning_rate"": 0.01, ""...",0.104,0.075,0.082,0.086,0.171
9,xgb,core + used cars + jet fuel,"{""n_estimators"": 300, ""learning_rate"": 0.01, ""...",0.104,0.075,0.080,0.085,0.174


## The naive rules and the poll on the same four blocks, for scale

In [4]:
bench = {"Last month's print": "core_last", "Average of the last 12 prints": "core_avg12", "Reuters poll": "poll"}
rows = []
for name, col in bench.items():
    scores = [rmse((dev.loc[a:b, "y"] - dev.loc[a:b, col]).dropna()) for a, b in FOLDS]
    rows.append({"benchmark": name, **dict(zip(fold_names, scores)), "cv_rmse": float(np.mean(scores))})
cv_bench = pd.DataFrame(rows).set_index("benchmark")
cv_bench.round(4).to_csv(TAB / "cv_benchmarks.csv")
cv_bench.round(3)

,2013-14,2015-16,2017-18,2019-20,cv_rmse
benchmark,,,,,
Last month's print,0.094,0.115,0.108,0.184,0.125
Average of the last 12 prints,0.077,0.076,0.091,0.195,0.110
Reuters poll,0.084,0.076,0.094,0.135,0.097


## The choice, fixed before any test month is used

For each model: the input set and hyperparameters with the lowest average block RMSE. Then the
robustness check Yi asked for: does the winner change if the 2019–20 block (the pandemic) is left out?

In [5]:
choice = {}
for kind in GRIDS:
    sub = cv[cv["model"] == kind]
    win = sub.loc[sub["cv_rmse"].idxmin()]
    alt = sub.loc[sub["cv_rmse_without_2019_20"].idxmin()]
    choice[kind] = {
        "label": MODEL_LABELS[kind],
        "feature_set": win["feature_set"],
        "params": json.loads(win["params"]),
        "cv_rmse": round(float(win["cv_rmse"]), 4),
        "fold_rmse": {f: round(float(win[f]), 4) for f in fold_names},
        "winner_without_2019_20": {"feature_set": alt["feature_set"], "params": json.loads(alt["params"])},
        "same_winner_without_2019_20": bool(alt.name == win.name),
    }
(TAB / "cv_choice.json").write_text(json.dumps(choice, indent=2))
pd.DataFrame({k: {"input set": v["feature_set"], "settings": v["params"], "CV RMSE": v["cv_rmse"],
                  "same winner without 2019-20": v["same_winner_without_2019_20"]} for k, v in choice.items()}).T

,input set,settings,CV RMSE,same winner without 2019-20
rf,core + jet fuel,"{'max_depth': 2, 'min_samples_leaf': 5, 'max_f...",0.1051,False
xgb,core + jet fuel,"{'n_estimators': 300, 'learning_rate': 0.01, '...",0.1035,False
ols,core + used cars,{},0.1012,False
